In [4]:
from pathlib import Path

dataset_dir = Path("../dataset/images")
image_exts = {".png"}

image_count = sum(1 for p in dataset_dir.rglob("*") if p.suffix.lower() in image_exts)
print(f"Total images in '{dataset_dir}': {image_count}")


Total images in '..\dataset\images': 150


In [5]:
from pathlib import Path
import json
import math

labels_dir = Path("../dataset/labels")
files = list(labels_dir.glob("*.json"))

fields = [
    "x",
    "y",
    "label",
    "topBarPixelDistance",
    "bottomBarPixelDistance",
    "deviationPixelDistance",
]

missing = {f: 0 for f in fields}
non_missing = {f: 0 for f in fields}
minmax = {f: [math.inf, -math.inf] for f in fields if f != "label"}

total_points = 0
for path in files:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        for p in item.get("points", []):
            total_points += 1
            for field in fields:
                if field not in p or p[field] is None:
                    missing[field] += 1
                    continue
                if field == "label" and str(p[field]).strip() == "":
                    missing[field] += 1
                    continue
                non_missing[field] += 1
                if field != "label":
                    try:
                        val = float(p[field])
                    except (TypeError, ValueError):
                        non_missing[field] -= 1
                        missing[field] += 1
                        continue
                    minmax[field][0] = min(minmax[field][0], val)
                    minmax[field][1] = max(minmax[field][1], val)

print(f"Label files: {len(files)}")
print(f"Total points: {total_points}")

for field in fields:
    print(f"\nField: {field}")
    print(f"  present: {non_missing[field]}")
    print(f"  missing: {missing[field]}")
    if field != "label" and non_missing[field] > 0:
        lo, hi = minmax[field]
        if math.isfinite(lo) and math.isfinite(hi):
            print(f"  range: {lo} to {hi}")


Label files: 150
Total points: 6815

Field: x
  present: 6815
  missing: 0
  range: 1.5384145883413451 to 2365.98779296875

Field: y
  present: 6815
  missing: 0
  range: 23.07059097290039 to 1396.9410400390625

Field: label
  present: 1976
  missing: 4839

Field: topBarPixelDistance
  present: 6815
  missing: 0
  range: 0.0 to 545.0

Field: bottomBarPixelDistance
  present: 6815
  missing: 0
  range: 0.0 to 537.0

Field: deviationPixelDistance
  present: 6815
  missing: 0
  range: 0.0 to 545.0


In [7]:
# Plot Structure + Visual Feature + Data Distribution + Statistical Metrics
from pathlib import Path
import json
import math
import numpy as np

images_dir = Path("../dataset/images")
labels_dir = Path("../dataset/labels")
image_paths = sorted(images_dir.glob("*.png"))
label_paths = sorted(labels_dir.glob("*.json"))

image_map = {p.stem: p for p in image_paths}
label_map = {p.stem: p for p in label_paths}
matched = sorted(set(image_map) & set(label_map))

def load_labels(path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def r2_score(xs, ys, coeffs):
    p = np.poly1d(coeffs)
    pred = p(xs)
    ss_res = np.sum((ys - pred) ** 2)
    ss_tot = np.sum((ys - np.mean(ys)) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")

# --- Plot Structure Analysis (from labels) ---
lines_per_plot = []
points_per_plot = []
x_ranges = []
y_ranges = []
x_spacings = []
y_spacings = []
outlier_counts = []

function_type_counts = {"linear": 0, "quadratic": 0, "other": 0}
trend_counts = {"increasing": 0, "decreasing": 0, "flat": 0}
noise_levels = []
error_bar_stats = []

for stem in matched:
    data = load_labels(label_map[stem])
    lines_per_plot.append(len(data))
    total_points = 0

    for item in data:
        pts = [(p.get("x"), p.get("y")) for p in item.get("points", [])]
        pts = [(x, y) for x, y in pts if x is not None and y is not None]
        total_points += len(pts)
        if len(pts) < 2:
            continue

        xs = np.array([p[0] for p in pts], dtype=float)
        ys = np.array([p[1] for p in pts], dtype=float)

        x_ranges.append((float(xs.min()), float(xs.max())))
        y_ranges.append((float(ys.min()), float(ys.max())))

        xs_sorted = np.sort(xs)
        ys_sorted = np.sort(ys)
        if len(xs_sorted) > 1:
            x_spacings.extend(np.diff(xs_sorted).tolist())
        if len(ys_sorted) > 1:
            y_spacings.extend(np.diff(ys_sorted).tolist())

        # Function type classification (simple): linear vs quadratic vs other
        coeffs1 = np.polyfit(xs, ys, 1)
        coeffs2 = np.polyfit(xs, ys, 2)
        r2_1 = r2_score(xs, ys, coeffs1)
        r2_2 = r2_score(xs, ys, coeffs2)
        if r2_1 >= 0.90:
            function_type_counts["linear"] += 1
        elif r2_2 >= 0.90:
            function_type_counts["quadratic"] += 1
        else:
            function_type_counts["other"] += 1

        # Trend analysis (using slope sign)
        slope = coeffs1[0]
        if slope > 0.01:
            trend_counts["increasing"] += 1
        elif slope < -0.01:
            trend_counts["decreasing"] += 1
        else:
            trend_counts["flat"] += 1

        # Noise level estimation (residual std / y std)
        pred = np.poly1d(coeffs2)(xs)
        resid = ys - pred
        noise = float(np.std(resid) / (np.std(ys) + 1e-9))
        noise_levels.append(noise)

        # Error bar characteristics
        for p in item.get("points", []):
            t = p.get("topBarPixelDistance")
            b = p.get("bottomBarPixelDistance")
            d = p.get("deviationPixelDistance")
            if t is not None or b is not None or d is not None:
                error_bar_stats.append((t or 0.0, b or 0.0, d or 0.0))

        # Outlier frequency (z-score > 3 on y)
        if len(ys) > 2:
            z = (ys - ys.mean()) / (ys.std() + 1e-9)
            outlier_counts.append(int((np.abs(z) > 3).sum()))

    points_per_plot.append(total_points)

# Distribution: lines per plot
lines_per_plot = np.array(lines_per_plot, dtype=float)
points_per_plot = np.array(points_per_plot, dtype=float)

print("Plot Structure Analysis")
if len(lines_per_plot) > 0:
    unique, counts = np.unique(lines_per_plot, return_counts=True)
    dist = dict(zip(unique.astype(int).tolist(), counts.tolist()))
    print("- Lines per plot distribution:", dist)

if len(points_per_plot) > 0:
    print(
        "- Data points density (min/avg/max):",
        f"{int(points_per_plot.min())}/{points_per_plot.mean():.2f}/{int(points_per_plot.max())}",
    )

single = int((lines_per_plot == 1).sum())
two = int((lines_per_plot == 2).sum())
three_plus = int((lines_per_plot >= 3).sum())
total_plots = max(int(len(lines_per_plot)), 1)
print(
    "- Plot ratio (single/two/three+):",
    f"{single/total_plots:.0%}/{two/total_plots:.0%}/{three_plus/total_plots:.0%}",
)

# --- Visual Feature Detection (from images) ---
print("\nVisual Feature Detection")
try:
    import cv2
    from sklearn.cluster import KMeans

    sample_imgs = image_paths[:20]
    line_counts = []
    grid_scores = []
    marker_counts = []
    palettes = []

    for p in sample_imgs:
        img = cv2.imread(str(p))
        if img is None:
            continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 50, 150)
        lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=80, minLineLength=50, maxLineGap=5)
        line_counts.append(0 if lines is None else len(lines))

        # Grid detection: count near-horizontal/vertical lines
        hv = 0
        if lines is not None:
            for x1, y1, x2, y2 in lines[:, 0]:
                dx = abs(x2 - x1)
                dy = abs(y2 - y1)
                if dx < 5 or dy < 5:
                    hv += 1
        grid_scores.append(hv)

        # Marker presence: simple blob detector
        params = cv2.SimpleBlobDetector_Params()
        params.filterByArea = True
        params.minArea = 10
        params.maxArea = 500
        detector = cv2.SimpleBlobDetector_create(params)
        keypoints = detector.detect(gray)
        marker_counts.append(len(keypoints))

        # Color palette via k-means on resized image
        small = cv2.resize(img, (128, 128), interpolation=cv2.INTER_AREA)
        pixels = small.reshape(-1, 3)
        kmeans = KMeans(n_clusters=5, n_init=5, random_state=0).fit(pixels)
        centers = kmeans.cluster_centers_.astype(int)
        palettes.append(centers.tolist())

    if line_counts:
        print("- Line style (approx, Hough count avg):", f"{np.mean(line_counts):.2f}")
    if marker_counts:
        print("- Marker presence (avg blobs):", f"{np.mean(marker_counts):.2f}")
    if grid_scores:
        print("- Grid presence score (avg HV lines):", f"{np.mean(grid_scores):.2f}")
    if palettes:
        print("- Dominant colors (sample 1):", palettes[0])
    print("- Legend/axis label positions: require OCR/layout analysis (not enabled here)")
except Exception as exc:
    print("Visual feature detection skipped (missing cv2/sklearn):", exc)

# --- Data Distribution Analysis ---
print("\nData Distribution Analysis")
print("- Function types:", function_type_counts)
print("- Trends:", trend_counts)
if noise_levels:
    print(
        "- Noise level (min/avg/max):",
        f"{min(noise_levels):.4f}/{np.mean(noise_levels):.4f}/{max(noise_levels):.4f}",
    )

if error_bar_stats:
    arr = np.array(error_bar_stats, dtype=float)
    print(
        "- Error bar stats (top/bottom/dev avg):",
        f"{arr[:,0].mean():.2f}/{arr[:,1].mean():.2f}/{arr[:,2].mean():.2f}",
    )

# --- Statistical Metrics ---
print("\nStatistical Metrics")
if x_ranges:
    xr = np.array(x_ranges)
    print(
        "- X-axis range (min..max):",
        f"{xr[:,0].min():.2f}..{xr[:,1].max():.2f}",
    )
if y_ranges:
    yr = np.array(y_ranges)
    print(
        "- Y-axis range (min..max):",
        f"{yr[:,0].min():.2f}..{yr[:,1].max():.2f}",
    )
if x_spacings:
    print(
        "- X spacing (min/avg/max):",
        f"{min(x_spacings):.2f}/{np.mean(x_spacings):.2f}/{max(x_spacings):.2f}",
    )
if y_spacings:
    print(
        "- Y spacing (min/avg/max):",
        f"{min(y_spacings):.2f}/{np.mean(y_spacings):.2f}/{max(y_spacings):.2f}",
    )
if outlier_counts:
    print(
        "- Outlier frequency (avg per line):",
        f"{np.mean(outlier_counts):.2f}",
    )


Plot Structure Analysis
- Lines per plot distribution: {1: 28, 2: 49, 3: 23, 4: 18, 5: 11, 6: 8, 7: 7, 9: 1, 10: 2, 14: 1, 15: 1, 17: 1}
- Data points density (min/avg/max): 8/45.43/290
- Plot ratio (single/two/three+): 19%/33%/49%

Visual Feature Detection
- Line style (approx, Hough count avg): 55.95
- Marker presence (avg blobs): 120.45
- Grid presence score (avg HV lines): 38.25
- Dominant colors (sample 1): [[254, 254, 254], [170, 170, 170], [228, 228, 228], [114, 114, 114], [201, 201, 201]]
- Legend/axis label positions: require OCR/layout analysis (not enabled here)

Data Distribution Analysis
- Function types: {'linear': 0, 'quadratic': 0, 'other': 494}
- Trends: {'increasing': 459, 'decreasing': 24, 'flat': 11}
- Noise level (min/avg/max): 0.6124/0.8945/0.9941
- Error bar stats (top/bottom/dev avg): 16.70/15.68/8.92

Statistical Metrics
- X-axis range (min..max): 1.54..2365.99
- Y-axis range (min..max): 23.07..1396.94
- X spacing (min/avg/max): 0.00/68.44/1364.64
- Y spacing (